# Steam Game Recommendation: Playtime-Aware BPR

**CSE 258, Assignment 2**

**Research question:** Does playtime-weighted Bayesian Personalized Ranking (BPR) outperform standard BPR on top-K game recommendation?

The notebook follows the five graded sections:

1. Predictive task and evaluation protocol
2. Dataset, preprocessing and exploratory analysis
3. Modeling
4. Evaluation and results
5. Related work and limitations

<!-- TEAM NOTES (delete before export)
Owners: [S] = Svante, [T] = Tony, [S+T] = both. Only edit cells in your own sections.
Pull before editing, commit small changes often, clear outputs before committing.
See doc.md for the full plan and integration contract. -->

## 0. Setup

In [ ]:
import ast
import gzip
import json
import math
import os
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")
for d in (DATA_DIR, RESULTS_DIR, FIGURES_DIR):
    d.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Predictive task and evaluation protocol  <!-- [S] -->

**Task.** Given a user's Steam library (which games they own and how long they have played each), rank the games they do not own so that the games they are most likely to own come first. The input is a user's observed library; the output is a score for every other game. The target is held-out game *ownership*, not future purchases, ratings or hours played: we hide part of each library and ask whether a model can recover it. This is top-K recommendation from implicit feedback, the setting of Bayesian Personalized Ranking (BPR).

**Research question.** Standard BPR treats every owned game as an equally strong positive. Does weighting each owned game by how long the user played it produce better rankings?

**Evaluation.** Each user's games are split at random into 80% training, 10% validation and 10% test. For every user, a model ranks all games not in their training library (for the test set, validation games are also excluded), and we compare the ranking with the held-out games:

- **Recall@10 and Recall@20:** the fraction of held-out games that appear in the top 10 or 20. A recommendation list is short, so what matters is whether the right games appear near the top.
- **NDCG@10** (primary metric): like Recall@10, but hits closer to rank 1 count more. It is used to select hyperparameters and to stop BPR training early.
- **AUC:** the probability that a held-out game is ranked above a random unowned game. It measures the whole ranking rather than the top, and is the metric BPR's objective approximates.

Metrics are computed per user and averaged, so heavy users with large libraries do not dominate.

**Baselines.**

1. **Popularity:** recommend the most-owned games to everyone. Any personalized model must beat this.
2. **Item-based Jaccard similarity:** recommend games often owned by the same people as the user's games. A simple, strong collaborative-filtering method with no training.
3. **Standard BPR:** the same model as our proposed method with all weights equal to 1. Comparing against it isolates the effect of playtime weighting.

**Validity.** Hyperparameters are chosen on validation data only, and the test set is used once for the final comparison. Every model is evaluated on the same split, the same candidate games and the same users, with fixed seeds and deterministic tie-breaking. The metric code is checked on a hand-computed example below. We also compare models across groups of users with different activity levels (Section 4.3) to see where any improvement comes from. One limitation applies to every model: a game a user does not own is not necessarily a game they would dislike, so these metrics measure agreement with observed libraries rather than true preference.

### 1.1 Evaluation functions  <!-- [S] -->

Defined here so they can be used for BPR early stopping in Section 3. Every model exposes `score_users(user_ids) -> np.ndarray` of shape `(len(user_ids), num_items)`, with scores in shared item-ID order.

For each user, `evaluate` ranks every game they do not own in training (for the test set, validation games are also excluded), then compares the ranking with the user's held-out games:

- **Recall@k:** the fraction of the user's held-out games that appear in the top k.
- **NDCG@10:** like Recall@10, but hits near the top count more (gain $1/\log_2(\text{rank}+1)$), normalized so a perfect ranking scores 1.
- **AUC:** the probability that a random held-out game is scored above a random game the user does not own, over the whole ranking.

Metrics are computed per user and then averaged, so every user counts equally regardless of library size. `sample_eval_users` gives a fixed user subset for fast evaluation during training.

In [ ]:
from scipy.stats import rankdata


def interaction_matrix(dfs, num_users, num_items):
    """Sparse 0/1 user x item matrix from one or more interaction DataFrames."""
    if isinstance(dfs, pd.DataFrame):
        dfs = [dfs]
    users = np.concatenate([df["user_id"].to_numpy() for df in dfs])
    items = np.concatenate([df["item_id"].to_numpy() for df in dfs])
    m = sp.csr_matrix((np.ones(len(users), dtype=np.float32), (users, items)),
                      shape=(num_users, num_items))
    m.data[:] = 1  # duplicates across DataFrames are summed by scipy; reset to 0/1
    return m


def sample_eval_users(eval_df, n_users, seed=SEED):
    """A fixed random subset of users, e.g. for fast per-epoch early stopping."""
    users = np.unique(eval_df["user_id"].to_numpy())
    return np.sort(np.random.default_rng(seed).choice(users, size=min(n_users, len(users)), replace=False))


def evaluate(model, eval_df, mask_dfs, num_users, num_items, ks=(10, 20), ndcg_k=10,
             user_ids=None, batch_size=1024, return_per_user=False):
    """Rank all train-supported items for each eligible user and compute ranking metrics.

    model:     object with score_users(user_ids) -> (len(user_ids), num_items) scores
    eval_df:   held-out interactions (val or test); a user's held-out games are its relevant items
    mask_dfs:  interactions excluded from ranking (train for val; [train, val] for test)
    user_ids:  optional subset of users to evaluate (e.g. from sample_eval_users)

    Eligible users are those with at least one held-out game. For each user:
      Recall@k  = (# held-out games in the top k) / (# held-out games)
      NDCG@10   = DCG@10 / ideal DCG@10, with gain 1 / log2(rank + 1)
      AUC       = probability that a random held-out game is scored above a random
                  candidate the user does not own (ties count 1/2)
    Top-k ties are broken by lower item ID, so results are deterministic.
    Returns macro averages over users, plus per-user metrics if return_per_user=True.
    """
    relevant = interaction_matrix(eval_df, num_users, num_items)
    masked = interaction_matrix(mask_dfs, num_users, num_items)
    assert relevant.multiply(masked).nnz == 0, "held-out items overlap the masked items"

    users = np.flatnonzero(np.diff(relevant.indptr) > 0)  # users with >= 1 held-out game
    if user_ids is not None:
        users = np.intersect1d(users, user_ids)

    max_k = min(max(max(ks), ndcg_k), num_items)
    discounts = 1.0 / np.log2(np.arange(2, max_k + 2))  # gain at rank 1, 2, ...
    ideal_dcg = np.cumsum(discounts[:ndcg_k])

    per_user = []
    for start in range(0, len(users), batch_size):
        batch = users[start:start + batch_size]
        scores = model.score_users(batch)
        if hasattr(scores, "detach"):  # accept torch tensors
            scores = scores.detach().cpu().numpy()
        scores = np.array(scores, dtype=np.float64)
        assert scores.shape == (len(batch), num_items), scores.shape

        rel = relevant[batch].toarray().astype(bool)
        mask = masked[batch].toarray().astype(bool)
        scores[mask] = -np.inf  # owned (training) games can never be recommended
        n_rel = rel.sum(axis=1)
        n_candidates = num_items - mask.sum(axis=1)

        # Top-k: stable sort on -score, so equal scores keep item-ID order.
        top = np.argsort(-scores, axis=1, kind="stable")[:, :max_k]
        hits = np.take_along_axis(rel, top, axis=1)
        row = {"user_id": batch, "n_relevant": n_rel}
        for k in ks:
            row[f"recall@{k}"] = hits[:, :k].sum(axis=1) / n_rel
        dcg = hits[:, :ndcg_k] @ discounts[:hits[:, :ndcg_k].shape[1]]
        row[f"ndcg@{ndcg_k}"] = dcg / ideal_dcg[np.minimum(n_rel, ndcg_k) - 1]

        # AUC via the Mann-Whitney statistic on average ranks (ties count 1/2).
        # Masked items are -inf, so they take the lowest m ranks; subtracting m gives
        # each item's rank among the candidates only.
        ranks = rankdata(scores, axis=1, method="average") - mask.sum(axis=1, keepdims=True)
        n_neg = n_candidates - n_rel
        rank_sum = np.where(rel, ranks, 0).sum(axis=1)
        row["auc"] = (rank_sum - n_rel * (n_rel + 1) / 2) / (n_rel * n_neg)
        per_user.append(pd.DataFrame(row))

    per_user = pd.concat(per_user, ignore_index=True)
    summary = per_user.drop(columns=["user_id", "n_relevant"]).mean().to_dict()
    summary["n_users"] = len(per_user)
    return (summary, per_user) if return_per_user else summary

In [ ]:
# Sanity check on a toy example small enough to verify by hand.
# One user, 6 games. Game 0 is owned in training (masked); games 1 and 4 are held out.
class FixedScores:
    def __init__(self, scores):
        self.scores = np.asarray(scores, dtype=float)

    def score_users(self, user_ids):
        return self.scores[user_ids]


toy_train = pd.DataFrame({"user_id": [0], "item_id": [0]})
toy_heldout = pd.DataFrame({"user_id": [0, 0], "item_id": [1, 4]})
# Candidate ranking by score: game 1 (rank 1), 2, 3, 4 (rank 4), 5. Game 0 is masked.
toy_model = FixedScores([[9.0, 0.9, 0.8, 0.7, 0.6, 0.5]])
toy = evaluate(toy_model, toy_heldout, [toy_train], num_users=1, num_items=6, ks=(1, 3))

# By hand: NDCG@10 = (1/log2(2) + 1/log2(5)) / (1/log2(2) + 1/log2(3)) = 0.877;
# AUC: game 1 beats all 3 negatives, game 4 beats 1 of 3 -> (3 + 1) / 6 = 0.667;
# Recall@1 = 1/2, Recall@3 = 1/2.
expected = {"recall@1": 0.5, "recall@3": 0.5, "ndcg@10": (1 + 1 / np.log2(5)) / (1 + 1 / np.log2(3)),
            "auc": 4 / 6}
for metric, value in expected.items():
    assert np.isclose(toy[metric], value), (metric, toy[metric], value)
print("Toy example matches hand-computed metrics:", {k: round(float(v), 3) for k, v in expected.items()})

## 2. Dataset, preprocessing and exploratory analysis  <!-- [S] -->

**Source.** We use the "User and Item Data" file from Version 1 of the Steam dataset (Pathak, Gupta and McAuley, 2017; Wan and McAuley, 2018; Kang and McAuley, 2018), available from Julian McAuley's dataset page (https://cseweb.ucsd.edu/~jmcauley/datasets.html#steam_data). Steam is Valve's digital storefront and launcher for PC games, where each user has a profile listing the games in their library.

**Contents.** The file has one record per user: their Steam ID, profile URL and the list of games they own. For each game it gives the lifetime playtime (`playtime_forever`, in minutes) and the playtime over the last two weeks. We use only the user ID, game ID and lifetime playtime.

**Collection.** Pathak et al. (2017) built the data by crawling the members of the "GameAus" group, an Australian Steam community, giving 88,310 users and 10,978 games. This matches the raw file exactly. A crawler can only see game libraries that users have made public, and the dataset itself is publicly released.

**Why this matters for the task.**
- *Ownership is imperfect implicit feedback.* Users own games they never play: games received in bundles or as gifts, free-to-play titles and sale purchases. A game missing from a library is not a dislike; the user may never have come across it. That is why the models treat unobserved games as sampled negatives rather than true negatives.
- *Playtime is a noisy preference signal.* Lifetime playtime is heavy-tailed and favours long, replayable games and games owned for longer. This motivates the log transform and per-user normalization tested in Section 3.4.
- *The data is a snapshot.* There are no purchase dates, so we cannot predict future purchases or split the data by time. We hold out a random subset of each user's library and ask whether a model can recover it.
- *Selection bias.* The users are members of a single Australian community group with public profiles, so they may not represent Steam users in general.

### 2.1 Download and parse  <!-- [S] -->

The file has one record per user with a nested list of owned games. We flatten it into one row per (user, game) with the game's lifetime playtime in minutes.

In [ ]:
# The raw file is committed at the repo root; download it only if it is missing.
RAW_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/steam/australian_users_items.json.gz"
RAW_PATH = Path("australian_users_items.json.gz")

if not RAW_PATH.exists():
    print("Downloading (~74 MB)...")
    urllib.request.urlretrieve(RAW_URL, RAW_PATH)

# Parsing takes ~1.5 minutes, so the flattened tables are cached in data/.
RAW_CACHE = DATA_DIR / "raw_interactions.parquet"
USERS_CACHE = DATA_DIR / "raw_users.parquet"
NAMES_CACHE = DATA_DIR / "item_names.parquet"


def parse_user_items(path):
    """Flatten the nested user -> items records into one row per (user, game).

    Each line of the file is a Python dict literal (single quotes), not strict JSON,
    so it is parsed with ast.literal_eval.
    """
    users, items, playtimes = [], [], []
    user_records = []
    item_names = {}
    with gzip.open(path, "rt") as f:
        for line in f:
            record = ast.literal_eval(line)
            user_records.append((record["user_id"], len(record["items"])))
            for it in record["items"]:
                users.append(record["user_id"])
                items.append(it["item_id"])
                playtimes.append(it["playtime_forever"])
                item_names[it["item_id"]] = it["item_name"]
    raw = pd.DataFrame({"user_id": users, "item_id": items, "playtime_forever": playtimes})
    raw_users = pd.DataFrame(user_records, columns=["user_id", "n_items"])
    names = pd.DataFrame(list(item_names.items()), columns=["item_id", "item_name"])
    return raw, raw_users, names


if RAW_CACHE.exists():
    raw = pd.read_parquet(RAW_CACHE)
    raw_users = pd.read_parquet(USERS_CACHE)
    item_names = pd.read_parquet(NAMES_CACHE)
else:
    raw, raw_users, item_names = parse_user_items(RAW_PATH)
    raw.to_parquet(RAW_CACHE, index=False)
    raw_users.to_parquet(USERS_CACHE, index=False)
    item_names.to_parquet(NAMES_CACHE, index=False)

display(pd.Series({
    "user records in file": len(raw_users),
    "unique users": raw_users["user_id"].nunique(),
    "user records with no games": int((raw_users["n_items"] == 0).sum()),
    "interaction rows": len(raw),
    "unique games": raw["item_id"].nunique(),
}, name="raw data").to_frame())
display(raw.head())

### 2.2 Cleaning and filtering  <!-- [S] -->

- **Duplicates.** Some users appear in more than one record, so a (user, game) pair can occur twice. Playtime only grows over time, so we keep the larger value.
- **Zero playtime.** Games that were never played are kept: owning a game is still an implicit signal of interest.
- **Non-game entries.** Some entries are not standalone games: test and beta builds that Steam adds to a library alongside the main game (e.g. *Dota 2 Test*, owned by 49,136 users, 93% of whom never played it), dedicated servers, SDKs and editors, soundtracks and demos. Owning them is not a choice worth modelling or recommending, and because they are almost never played they would also bias the comparison between standard and playtime-weighted BPR. We remove them by name, never by playtime, keeping a short list of real games whose names happen to match (e.g. *Test Drive Unlimited 2*).
- **Filtering.** We keep users with at least 10 games and games with at least 5 users, repeating until stable, because removing games can push users below the threshold and vice versa. Very sparse users and games give the models too little signal to learn from or be evaluated on.

In [ ]:
MIN_GAMES_PER_USER = 10
MIN_USERS_PER_GAME = 5


def summarize(df):
    """Dataset size statistics for one stage of preprocessing."""
    n_users, n_items = df["user_id"].nunique(), df["item_id"].nunique()
    return {
        "users": n_users,
        "games": n_items,
        "interactions": len(df),
        "density": len(df) / (n_users * n_items),
        "zero-playtime fraction": (df["playtime_forever"] == 0).mean(),
    }


# 1. Duplicates: some users appear in more than one record, so the same (user, game)
#    pair can occur twice. playtime_forever is cumulative, so keep the larger value.
pair_playtimes = raw.groupby(["user_id", "item_id"])["playtime_forever"]
n_conflicting = int((pair_playtimes.nunique() > 1).sum())
clean = pair_playtimes.max().reset_index()
print(f"Removed {len(raw) - len(clean):,} duplicate rows "
      f"({n_conflicting} duplicated pairs had different playtimes; kept the max).")

# 2. Validate playtime and convert minutes to hours. Zero-playtime games are kept:
#    owning a game is still an implicit signal even if it was never played.
assert (clean["playtime_forever"] >= 0).all(), "negative playtime found"
clean["playtime_hours"] = clean["playtime_forever"] / 60


# 3. Remove entries that are not standalone games: test/beta builds that Steam grants
#    alongside a game, dedicated servers, SDKs and editors, soundtracks and demos.
#    Owning these is not a choice we want to model or recommend. The filter uses names
#    only (never playtime), with an explicit list of real games whose names match.
NON_GAME_PATTERN = (r"test server|public test|patch testing|system test|\btest\b|\bbeta\b|\bunstable\b"
                    r"|dedicated server|\bsdk\b|\beditor\b|authoring tools|soundtrack|\bdemo\b")
NON_GAME_EXCEPTIONS = {
    "9930",    # Test Drive Unlimited 2
    "211970",  # Test Drive: Ferrari Racing Legends
    "499520",  # The Turing Test
    "430290",  # Shmups Skill Test
    "273550",  # Terrain Test
    "452040",  # GameLoading: Beta The Robot (documentary)
    "490980",  # Daily Chthonicle: Editor's Edition
    "238590",  # Loadout Campaign Beta (standalone free game)
    "270370",  # Lambda Wars Beta (standalone free game)
    "219700",  # Dogs of War Online - Beta (standalone game)
    "313240",  # Wilson Chronicles - Beta (standalone game)
    "457550",  # Bigscreen Beta (standalone app)
}
is_non_game = (item_names["item_name"].str.contains(NON_GAME_PATTERN, case=False, regex=True)
               & ~item_names["item_id"].isin(NON_GAME_EXCEPTIONS))
non_games = item_names.loc[is_non_game, "item_id"]
removed = clean[clean["item_id"].isin(non_games)]
print(f"Removed {non_games.nunique()} non-game entries ({len(removed):,} interactions, "
      f"{len(removed) / len(clean):.1%} of the total). Most-owned removed entries:")
display(removed.merge(item_names, on="item_id").groupby("item_name").size()
        .sort_values(ascending=False).head(10).rename("owners").to_frame())
before_non_game = summarize(clean)
clean = clean[~clean["item_id"].isin(non_games)]


# 4. Iterative k-core filtering: removing games can push users below the threshold
#    and vice versa, so repeat until nothing changes.
def iterative_filter(df, min_games_per_user, min_users_per_game):
    history = [summarize(df)]
    while True:
        n_before = len(df)
        df = df[df.groupby("user_id")["item_id"].transform("size") >= min_games_per_user]
        df = df[df.groupby("item_id")["user_id"].transform("size") >= min_users_per_game]
        history.append(summarize(df))
        if len(df) == n_before:
            break
    return df.reset_index(drop=True), pd.DataFrame(history)


filtered, filter_history = iterative_filter(clean, MIN_GAMES_PER_USER, MIN_USERS_PER_GAME)
filter_history = pd.concat([pd.DataFrame([before_non_game]), filter_history], ignore_index=True)
filter_history.index = (["after dedup", "after removing non-games"]
                        + [f"filter pass {i}" for i in range(1, len(filter_history) - 1)])
display(filter_history)

### 2.3 Train / validation / test split  <!-- [S] -->

Each user's games are randomly split 80/10/10 (seeded), with at least one validation and one test game per user. The dataset is a snapshot with no purchase dates, so a random split is used rather than a chronological one. Games are indexed from the training split only, since a model cannot score a game it never saw in training.

The split is saved to `data/` so the model sections can reload it without rerunning preprocessing.

In [ ]:
HELDOUT_FRACTION = 0.1  # per user: 10% validation, 10% test, 80% train


def per_user_split(df, seed, heldout_fraction=HELDOUT_FRACTION):
    """Randomly assign each user's games to train/val/test.

    Each user gets n_held = max(1, round(0.1 * n)) validation games and the same number
    of test games; the rest are training games. Users have >= 10 games after filtering,
    so every user has at least 8 training games.
    """
    rng = np.random.default_rng(seed)
    df = df.assign(_r=rng.random(len(df))).sort_values(["user_id", "_r"], kind="stable")
    position = df.groupby("user_id").cumcount().to_numpy()
    n = df.groupby("user_id")["item_id"].transform("size").to_numpy()
    n_held = np.maximum(1, np.floor(heldout_fraction * n + 0.5)).astype(int)
    split = np.where(position < n_held, "test",
                     np.where(position < 2 * n_held, "val", "train"))
    return df.drop(columns="_r").assign(split=split)


split_df = per_user_split(filtered, SEED)

# Contiguous integer IDs. Items are indexed from the training split only: a model cannot
# score a game it never saw in training, so held-out rows for such games are excluded.
user_index = pd.Index(np.sort(split_df["user_id"].unique()))
item_index = pd.Index(np.sort(split_df.loc[split_df["split"] == "train", "item_id"].unique()))

split_df["uid"] = user_index.get_indexer(split_df["user_id"])
split_df["iid"] = item_index.get_indexer(split_df["item_id"])  # -1 = not in training
unsupported = split_df["iid"] < 0
print(f"Excluded {unsupported.sum():,} held-out interactions whose game never appears in training.")
split_df = split_df[~unsupported]


def to_output(part):
    return pd.DataFrame({
        "user_id": part["uid"].astype("int32").to_numpy(),
        "item_id": part["iid"].astype("int32").to_numpy(),
        "playtime_hours": part["playtime_hours"].astype("float32").to_numpy(),
    }).sort_values(["user_id", "item_id"], ignore_index=True)


splits = {name: to_output(split_df[split_df["split"] == name]) for name in ("train", "val", "test")}
num_users, num_items = len(user_index), len(item_index)

# Sanity checks: every user has training data, IDs are contiguous, partitions are disjoint.
assert splits["train"]["user_id"].nunique() == num_users
assert splits["train"]["item_id"].nunique() == num_items
pair_keys = {name: set(zip(s["user_id"], s["item_id"])) for name, s in splits.items()}
assert not (pair_keys["train"] & pair_keys["val"] or pair_keys["train"] & pair_keys["test"]
            or pair_keys["val"] & pair_keys["test"])

total = sum(len(s) for s in splits.values())
display(pd.DataFrame([
    {
        "split": name,
        "interactions": len(s),
        "share": round(len(s) / total, 4),
        "users with >= 1 interaction": s["user_id"].nunique(),
        "users with none (ineligible)": num_users - s["user_id"].nunique(),
    } for name, s in splits.items()
]).set_index("split"))

# Save the shared split (the interface between data/evaluation and model code).
for name, s in splits.items():
    s.to_parquet(DATA_DIR / f"{name}.parquet", index=False)

names_by_id = item_names.set_index("item_id")["item_name"]
meta = {
    "num_users": num_users,
    "num_items": num_items,
    "seed": SEED,
    "heldout_fraction": HELDOUT_FRACTION,
    "min_games_per_user": MIN_GAMES_PER_USER,
    "min_users_per_game": MIN_USERS_PER_GAME,
    "excluded_heldout_interactions": int(unsupported.sum()),
    "user_ids": user_index.tolist(),  # position = integer user_id
    "item_ids": item_index.tolist(),  # position = integer item_id
    "item_names": names_by_id.reindex(item_index).fillna("").tolist(),
}
with open(DATA_DIR / "meta.json", "w") as f:
    json.dump(meta, f)
print(f"Saved train/val/test parquet files and meta.json: {num_users:,} users, {num_items:,} games.")

### 2.4 Exploratory analysis  <!-- [S] -->

Statistics are computed on the filtered dataset (all splits combined). They describe the data only; no model uses them.

In [ ]:
# Descriptive statistics of the filtered dataset (all splits combined; no model uses them).
eda = filtered.merge(item_names, on="item_id", how="left")
games_per_user = eda.groupby("user_id").size()
users_per_game = eda.groupby("item_id").size()
hours = eda["playtime_hours"]

n_users, n_games = len(games_per_user), len(users_per_game)
display(pd.Series({
    "users": f"{n_users:,}",
    "games": f"{n_games:,}",
    "interactions": f"{len(eda):,}",
    "sparsity": f"{1 - len(eda) / (n_users * n_games):.4%}",
    "games per user (median / mean / max)":
        f"{games_per_user.median():.0f} / {games_per_user.mean():.1f} / {games_per_user.max():,}",
    "users per game (median / mean / max)":
        f"{users_per_game.median():.0f} / {users_per_game.mean():.1f} / {users_per_game.max():,}",
    "zero-playtime fraction": f"{(hours == 0).mean():.1%}",
}, name="filtered dataset").to_frame())

quantiles = [0.25, 0.5, 0.75, 0.9, 0.99, 1.0]
display(pd.DataFrame({
    "all interactions": hours.quantile(quantiles),
    "played (> 0 h)": hours[hours > 0].quantile(quantiles),
}).rename(index=lambda q: "max" if q == 1 else f"{q:.0%}").round(1)
  .rename_axis("playtime quantile (hours)"))

# Most-owned games vs games with the most total playtime: popularity and engagement differ.
by_game = eda.groupby(["item_id", "item_name"]).agg(
    owners=("user_id", "size"),
    total_hours=("playtime_hours", "sum"),
    zero_playtime_share=("playtime_hours", lambda h: (h == 0).mean()),
).reset_index(level=0, drop=True)
top_owned = by_game.sort_values("owners", ascending=False).head(10)
top_played = by_game.sort_values("total_hours", ascending=False).head(10)
display(top_owned.style.format({"total_hours": "{:,.0f}", "zero_playtime_share": "{:.0%}"})
        .set_caption("Top 10 games by number of owners"))
display(top_played.style.format({"total_hours": "{:,.0f}", "zero_playtime_share": "{:.0%}"})
        .set_caption("Top 10 games by total playtime"))

# Does a bigger library mean more total playtime? Relevant to the activity-group
# analysis in Section 4.3, which groups users by total playtime.
user_totals = eda.groupby("user_id")["playtime_hours"].sum()
rho = games_per_user.rank().corr(user_totals.rank())
print(f"Spearman correlation between library size and total playtime: {rho:.2f}")

In [ ]:
import matplotlib.ticker  # noqa: E402  (tick formatting)

# Plot style: one series per panel, recessive axes and grid.
BLUE, INK, MUTED, GRID = "#2a78d6", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
})


def log_hist(ax, values, xlabel, title):
    """Histogram with log-spaced bins on log-log axes, for long-tailed counts."""
    bins = np.logspace(np.log10(values.min()), np.log10(values.max()), 40)
    ax.hist(values, bins=bins, color=BLUE, edgecolor="white", linewidth=0.6)
    ax.set(xscale="log", yscale="log", xlabel=xlabel, ylabel="count", title=title)
    ax.axvline(values.median(), color=INK, linestyle="--", linewidth=1)
    ax.annotate(f"median {values.median():.0f}", xy=(values.median(), 1),
                xycoords=("data", "axes fraction"), xytext=(4, -14),
                textcoords="offset points", color=INK, fontsize=9)


fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
log_hist(axes[0], games_per_user, "games in library", "Games per user")
log_hist(axes[1], users_per_game, "number of owners", "Users per game")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_degree_distributions.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(hours, bins=60, color=BLUE, edgecolor="white", linewidth=0.6)
axes[0].set(yscale="log", xlabel="playtime (hours)", ylabel="interactions (log scale)",
            title="Raw playtime")
played = hours[hours > 0]
axes[1].hist(np.log1p(played), bins=60, color=BLUE, edgecolor="white", linewidth=0.6)
axes[1].set(xlabel="log(1 + playtime in hours)", ylabel="interactions",
            title="Log-transformed playtime (played games only)")
axes[1].yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
axes[1].text(0.98, 0.95, f"not shown: {(hours == 0).sum():,} zero-playtime\ninteractions ({(hours == 0).mean():.0%})",
             transform=axes[1].transAxes, ha="right", va="top", color=MUTED, fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_playtime.png", bbox_inches="tight")
plt.show()

**Observations.**

- *Long tails on both sides.* The median user owns 52 games, but some own thousands; the median game has 70 owners, while the most popular have over 40,000. Popularity alone will be a strong baseline, and models must do better than recommending the same few games to everyone.
- *Playtime is extremely skewed.* 35% of owned games were never played, the median is under an hour, and the top 1% exceed ~300 hours. Raw hours would let a handful of interactions dominate a weighted loss; after `log(1 + t)` the played interactions spread out smoothly, which motivates the log weighting in Section 3.4.
- *Ownership is not the same as engagement.* The most-owned and most-played games overlap only partly: *Unturned* and *Portal 2* are among the most owned but not the most played, while *Skyrim* and *Civilization V* rank higher by playtime than by owners. Free-to-play games are owned widely but played by fewer of their owners (10% of *Unturned* owners and 18% of *Warframe* owners never played them).
- *Activity is confounded with library size.* Users with larger libraries also play more in total (Spearman correlation 0.60), so the activity groups in Section 4.3 partly reflect library size.

## 3. Modeling  <!-- [S+T] -->

TODO: formulation (inputs, outputs, objective) and the trade-offs between the models (complexity, efficiency, implementation challenges).

### 3.0 Load the shared split  <!-- [S] -->

In [ ]:
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
val_df = pd.read_parquet(DATA_DIR / "val.parquet")
test_df = pd.read_parquet(DATA_DIR / "test.parquet")
with open(DATA_DIR / "meta.json") as f:
    meta = json.load(f)
num_users, num_items = meta["num_users"], meta["num_items"]

### 3.1 Popularity baseline  <!-- [S] -->

Score each game by its number of distinct training users, so every user gets the same ranking (minus the games they already own). It ignores personal taste entirely, which makes it the trivial baseline any personalized model must beat.

In [ ]:
class Popularity:
    """Score every game by its number of distinct training users (the same list for everyone)."""

    def __init__(self, train_df, num_items):
        self.counts = np.bincount(train_df["item_id"], minlength=num_items).astype(np.float32)

    def score_users(self, user_ids):
        return np.broadcast_to(self.counts, (len(user_ids), len(self.counts)))


popularity = Popularity(train_df, num_items)
EARLY_STOP_USERS = sample_eval_users(val_df, 5000)  # fixed subset for quick validation checks
print("Popularity (validation, 5,000 users):",
      {k: round(v, 4) for k, v in evaluate(popularity, val_df, [train_df], num_users, num_items,
                                           user_ids=EARLY_STOP_USERS).items()})

### 3.2 Item-based Jaccard similarity  <!-- [S] -->

$J(i,j)=\frac{|U_i\cap U_j|}{|U_i\cup U_j|}$, where $U_i$ is the set of training users who own game $i$. A candidate's score is the sum of its similarities to the games in the user's training history: games often owned by the same people as the user's games rank highly.

Similarities are computed in blocks of games with sparse matrix products, so the full user-by-user structure is never materialized. Each game can keep only its $k$ most similar games (ties broken by lower game ID); we choose $k$ on the validation users below.

In [ ]:
class ItemJaccard:
    """Item-based collaborative filtering with Jaccard similarity.

    J(i, j) = |U_i & U_j| / |U_i | U_j|, where U_i is the set of training users who own game i.
    Each game keeps only its `top_k_neighbors` most similar games (None keeps all; ties broken
    by lower game ID), and a user's score for game j is the sum of J(i, j) over the games i in
    their training library.
    """

    def __init__(self, train_df, num_users, num_items, top_k_neighbors=100, block_size=1024):
        # items x users ownership matrix
        X = sp.csr_matrix((np.ones(len(train_df), dtype=np.float32),
                           (train_df["item_id"], train_df["user_id"])), shape=(num_items, num_users))
        n_owners = np.asarray(X.sum(axis=1)).ravel()
        XT = X.T.tocsc()
        prune = top_k_neighbors is not None and top_k_neighbors < num_items
        # Pruned: sparse (top-k per row). All neighbours: ~80% of pairs are nonzero, so dense
        # float32 (num_items^2 * 4 bytes, ~330 MB) is smaller and faster than sparse.
        S = None if prune else np.zeros((num_items, num_items), dtype=np.float32)
        rows, cols, vals = [], [], []
        # Compute similarities one block of games at a time to bound memory.
        for start in range(0, num_items, block_size):
            block = np.arange(start, min(start + block_size, num_items))
            co = (X[block] @ XT).toarray()  # |U_i & U_j|
            sim = co / (n_owners[block, None] + n_owners[None, :] - co)  # Jaccard
            sim[np.arange(len(block)), block] = 0  # a game is not its own neighbour
            if prune:
                keep = np.argsort(-sim, axis=1, kind="stable")[:, :top_k_neighbors]
                kept = np.take_along_axis(sim, keep, axis=1)
                nonzero = kept > 0
                rows.append(np.repeat(block, nonzero.sum(axis=1)))
                cols.append(keep[nonzero])
                vals.append(kept[nonzero].astype(np.float32))
            else:
                S[block] = sim
        if prune:  # S[i, j] = J(i, j) if j is among game i's kept neighbours, else 0
            S = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
                              shape=(num_items, num_items))
        self.S = S
        self.R = sp.csr_matrix((np.ones(len(train_df), dtype=np.float32),
                                (train_df["user_id"], train_df["item_id"])), shape=(num_users, num_items))

    def score_users(self, user_ids):
        scores = self.R[user_ids] @ self.S  # sum of similarities to the user's games
        return scores.toarray() if sp.issparse(scores) else np.asarray(scores)

In [ ]:
# Choose the neighbourhood size on validation data (fixed user subset).
jaccard_sweep = []
for k in (25, 100, 400, None):
    t0 = time.time()
    model = ItemJaccard(train_df, num_users, num_items, top_k_neighbors=k)
    fit_seconds = time.time() - t0
    metrics = evaluate(model, val_df, [train_df], num_users, num_items, user_ids=EARLY_STOP_USERS)
    jaccard_sweep.append({"top_k_neighbors": "all" if k is None else k, **metrics,
                          "fit seconds": round(fit_seconds, 1)})
jaccard_sweep = pd.DataFrame(jaccard_sweep).set_index("top_k_neighbors").drop(columns="n_users")
display(jaccard_sweep.round(4))

best_k = jaccard_sweep["ndcg@10"].idxmax()
jaccard = ItemJaccard(train_df, num_users, num_items,
                      top_k_neighbors=None if best_k == "all" else int(best_k))
print(f"Selected top_k_neighbors = {best_k} (highest validation NDCG@10).")

### 3.3 Standard BPR  <!-- [T] -->

BPR scores game $i$ for user $u$ with the dot product $s(u,i)=p_u^\top q_i$. In each epoch, we shuffle the eligible training interactions and sample one negative per positive, uniformly from the games observed in training that the user did not own in the training split. A user who owns the entire training catalog has no valid negative and contributes no triplets.

We optimize the mean triplet loss with Adam:

$$
\mathcal L = \mathbb E_{(u,i,j)}\left[
\operatorname{softplus}\big(s(u,j)-s(u,i)\big)
+ \lambda\big(\|p_u\|_2^2 + \|q_i\|_2^2 + \|q_j\|_2^2\big)
\right].
$$

`train_bpr(train_df, val_df, num_users, num_items)` returns the fitted model and per-epoch history. When given a validation split, it selects the checkpoint with the highest NDCG@10 and stops after five epochs without improvement by default. The model's `score_users` method returns a row of scores per requested user in shared item-ID order. The shared evaluator handles candidate masking.

In [ ]:
class BPR(torch.nn.Module):
    def __init__(self, num_users, num_items, dim=64):
        super().__init__()
        self.user_embeddings = torch.nn.Embedding(num_users, dim)
        self.item_embeddings = torch.nn.Embedding(num_items, dim)
        torch.nn.init.normal_(self.user_embeddings.weight, std=0.01)
        torch.nn.init.normal_(self.item_embeddings.weight, std=0.01)

    @torch.no_grad()
    def score_users(self, user_ids):
        """Score every item in the shared item-ID order."""
        ids = torch.as_tensor(user_ids, dtype=torch.long,
                              device=self.user_embeddings.weight.device)
        scores = self.user_embeddings(ids) @ self.item_embeddings.weight.T
        return scores.cpu().numpy()


def _bpr_validation_scorer(train_df, val_df, num_users, num_items, batch_size=512):
    """Compute the shared evaluator's NDCG@10 on every eligible user."""
    relevant = interaction_matrix(val_df, num_users, num_items).astype(bool)
    masked = interaction_matrix(train_df, num_users, num_items)
    if relevant.multiply(masked).nnz:
        raise ValueError("Validation interactions overlap the training mask.")

    counts = np.diff(relevant.indptr)
    users = np.flatnonzero(counts)
    if not len(users):
        raise ValueError("BPR checkpoint selection requires validation interactions.")
    k = min(10, num_items)
    discounts = 1 / np.log2(np.arange(2, k + 2))
    ideal = np.cumsum(discounts)[np.minimum(counts[users], k) - 1]
    batches = []
    for start in range(0, len(users), batch_size):
        batch = users[start:start + batch_size]
        batches.append((batch, masked[batch].nonzero(), relevant[batch]))

    def ndcg_at_10(model):
        dcg = np.empty(len(users), dtype=np.float64)
        for start, (batch, mask, truth) in enumerate(batches):
            scores = model.score_users(batch)
            scores[mask] = -np.inf
            values, indices = torch.topk(torch.from_numpy(scores), min(k + 1, num_items))
            values, top = values.numpy(), indices.numpy()[:, :k]
            # Include one extra score to detect ties at the cutoff.
            tied_rows = np.flatnonzero((values[:, 1:] == values[:, :-1]).any(axis=1))
            for row in tied_rows:
                top[row] = np.argsort(-scores[row], kind="stable")[:k]
            hits = np.take_along_axis(truth.toarray(), top, axis=1)
            offset = start * batch_size
            dcg[offset:offset + len(batch)] = hits @ discounts
        return float(np.mean(dcg / ideal))

    return ndcg_at_10


def train_bpr(train_df, val_df, num_users, num_items, weights=None, dim=64, lr=1e-3,
              reg=1e-4, batch_size=4096, epochs=30, seed=SEED, patience=5, verbose=False, fused_adam=False):
    """Return the model and per-epoch training history.

    Select checkpoints by full-catalog validation NDCG@10. Pass val_df=None
    to train for a fixed number of epochs.
    """
    users = train_df["user_id"].to_numpy(dtype=np.int64)
    items = train_df["item_id"].to_numpy(dtype=np.int64)
    if not len(users):
        raise ValueError("BPR requires training interactions.")
    if (users.min() < 0 or users.max() >= num_users
            or items.min() < 0 or items.max() >= num_items):
        raise ValueError("Training IDs must be within the shared ID ranges.")
    if min(dim, batch_size, epochs, patience) < 1:
        raise ValueError("dim, batch_size, epochs and patience must be positive.")
    if weights is None:
        weights = np.ones(len(users), dtype=np.float32)
    else:
        weights = np.asarray(weights, dtype=np.float32)
        if (weights.shape != users.shape or not np.isfinite(weights).all()
                or np.any(weights < 0)):
            raise ValueError("Provide one finite, nonnegative weight per training row.")

    # Work in the train-supported catalog so held-out-only items cannot
    # influence training. Keep the original ID order for model scores.
    catalog, item_ranks = np.unique(items, return_inverse=True)
    catalog_size = len(catalog)
    seen = np.unique(users * catalog_size + item_ranks)
    seen_users = seen // catalog_size
    seen_counts = np.bincount(seen_users, minlength=num_users)
    row_starts = np.cumsum(seen_counts) - seen_counts
    available = catalog_size - seen_counts
    eligible_rows = np.flatnonzero(available[users])
    if not len(eligible_rows):
        raise ValueError("No training user has an unowned game to sample.")

    # Subtract the number of prior owned items so searchsorted can map
    # a uniformly sampled unowned rank directly to its item, without retries.
    seen_offsets = np.arange(len(seen)) - row_starts[seen_users]
    gap_keys = seen - seen_offsets

    sampling_rng = np.random.default_rng(seed)
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(seed)
        model = BPR(num_users, num_items, dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, fused=fused_adam)

    score_validation = (
        _bpr_validation_scorer(train_df, val_df, num_users, num_items)
        if val_df is not None else None
    )
    history = []
    best_score = -np.inf
    best_state = None
    stale_epochs = 0
    for epoch in range(1, epochs + 1):
        model.train()
        shuffled_rows = sampling_rng.permutation(eligible_rows)
        total_loss = 0.0
        for start in range(0, len(shuffled_rows), batch_size):
            rows = shuffled_rows[start:start + batch_size]
            batch_users = users[rows]
            draws = sampling_rng.integers(available[batch_users])
            queries = batch_users * catalog_size + draws
            skipped = np.searchsorted(gap_keys, queries, side="right")
            negative_ranks = draws + skipped - row_starts[batch_users]

            user_vectors = model.user_embeddings(torch.from_numpy(batch_users))
            positive_vectors = model.item_embeddings(torch.from_numpy(items[rows]))
            negative_vectors = model.item_embeddings(torch.from_numpy(catalog[negative_ranks]))
            margins = (user_vectors * (positive_vectors - negative_vectors)).sum(dim=1)
            ranking_loss = (torch.from_numpy(weights[rows])
                            * torch.nn.functional.softplus(-margins)).mean()
            penalty = (user_vectors.square().sum(dim=1)
                       + positive_vectors.square().sum(dim=1)
                       + negative_vectors.square().sum(dim=1)).mean()
            loss = ranking_loss + reg * penalty

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(rows)

        record = {"epoch": epoch, "train_loss": total_loss / len(shuffled_rows)}
        history.append(record)
        if score_validation is not None:
            model.eval()
            score = score_validation(model)
            if not np.isfinite(score):
                raise ValueError("Validation NDCG@10 must be finite.")
            record["val_ndcg@10"] = score
            if verbose:
                print(f"Epoch {epoch:2d} | loss {record['train_loss']:.6f} | "
                      f"validation NDCG@10 {score:.6f}", flush=True)
            if score > best_score:
                best_score = score
                best_state = {name: value.clone() for name, value in model.state_dict().items()}
                stale_epochs = 0
            else:
                stale_epochs += 1
                if stale_epochs >= patience:
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model.eval(), history

### 3.4 Playtime-weighted BPR  <!-- [T] -->

For training playtime $t_{ui}$ in hours, define $a_{ui}=(1+\log(1+t_{ui}))^\alpha$. We multiply the pairwise softplus term from Section 3.3 by $w_{ui}$ and use the same sampled L2 penalty, embeddings, and negative sampler across variants.

| Variant | Raw weight |
|---|---|
| Standard BPR | 1 |
| Log-weighted BPR | $a_{ui}$ |
| User-normalized BPR | $a_{ui}/\operatorname{mean}_{k\in I_u^{train}}a_{uk}$ |

We normalize weights to mean one over interactions eligible for triplet sampling. User normalization also gives each user's interactions mean weight one, preserving that user's total weight from standard BPR. A game with zero recorded playtime has a raw weight of one before normalization. Setting $\alpha=0$ gives standard BPR.

In [ ]:
def playtime_weights(train_df, alpha=1.0, user_normalize=False):
    """Return row-aligned weights with mean one over trainable triplets."""
    hours = train_df["playtime_hours"].to_numpy(dtype=np.float64)
    if not np.isfinite(alpha) or alpha < 0:
        raise ValueError("alpha must be finite and nonnegative.")
    if not np.isfinite(hours).all() or np.any(hours < 0):
        raise ValueError("Training playtime must be finite and nonnegative.")

    weights = np.ones(len(hours), dtype=np.float32)
    if alpha == 0 or not len(hours):
        return weights

    # Subtracting a common log scale before exponentiating preserves the
    # normalized weights and keeps the largest exponent at zero.
    log_base = np.log1p(np.log1p(hours))
    if user_normalize:
        users = train_df["user_id"].to_numpy()
        maxima = pd.Series(log_base).groupby(users, sort=False).transform("max").to_numpy()
        scaled = np.exp(alpha * (log_base - maxima))
        means = pd.Series(scaled).groupby(users, sort=False).transform("mean").to_numpy()
        return (scaled / means).astype(np.float32)

    owned = train_df.groupby("user_id", sort=False)["item_id"].transform("nunique")
    eligible = owned.to_numpy() < train_df["item_id"].nunique()
    if eligible.any():
        eligible_log_base = log_base[eligible]
        scaled = np.exp(alpha * (eligible_log_base - eligible_log_base.max()))
        weights[eligible] = scaled / scaled.mean()
    return weights


def run_alpha_sweep(train_df, val_df, num_users, num_items,
                    alphas=(0, 0.25, 0.5, 1, 2), previous_runs=None, **bpr_config):
    """Return epoch records, the selected alpha, and its fitted model."""
    records = []
    best_alpha, best_model, best_score = None, None, -np.inf
    previous_runs = {} if previous_runs is None else previous_runs
    for alpha in sorted(alphas):
        if alpha in previous_runs:
            model, history = previous_runs[alpha]
        else:
            weights = playtime_weights(train_df, alpha=alpha)
            model, history = train_bpr(
                train_df, val_df, num_users, num_items, weights=weights, **bpr_config
            )
        run_best = max(history, key=lambda row: row["val_ndcg@10"])
        if run_best["val_ndcg@10"] > best_score:
            best_alpha, best_model, best_score = alpha, model, run_best["val_ndcg@10"]
        records.extend({
            "model": "standard_bpr" if alpha == 0 else "log_weighted_bpr",
            "split": "validation",
            "alpha": alpha,
            **bpr_config,
            **row,
        } for row in history)
    return pd.DataFrame(records), best_alpha, best_model

## 4. Evaluation and results  <!-- [S+T] -->

Recall@10 and Recall@20 measure how many held-out games appear in the recommendation list. NDCG@10 rewards putting held-out games near the top, where users are most likely to see them. AUC summarizes how consistently relevant games outrank other candidates across the catalog. The test evaluator masks each user's training and validation games and macro-averages across eligible users. Checkpoint and hyperparameter selection use validation data.

### 4.1 Experiment A: main comparison (test set)  <!-- [S+T] -->

The BPR variants train for up to 100 epochs and restore the best validation NDCG@10 checkpoint after five epochs without improvement. The alpha sweep also uses this budget. The test evaluator runs after validation has selected the checkpoints and alpha. Jaccard uses all neighbors, which gave the highest NDCG@10 when tuning on all validation users.

Standard BPR led all five models on NDCG@10 (0.2790), Recall@10 (0.2603), Recall@20 (0.3567), and AUC (0.9772). With the longer training budget, its test NDCG@10 rose from 0.2498 to 0.2790 and exceeded Jaccard's 0.2702. The log-weighted and user-normalized variants reached 0.2586 and 0.2524. Both also improved with longer training, and standard BPR remained the strongest BPR variant in this single-seed experiment.

In [ ]:
def evaluate_test_models(models, train_df, val_df, test_df, num_users, num_items):
    """Evaluate fixed models on the same test users and candidate masks."""
    rows, per_user = [], {}
    for name, (model, configuration) in models.items():
        metrics, per_user[name] = evaluate(
            model, test_df, [train_df, val_df], num_users, num_items,
            batch_size=512, return_per_user=True,
        )
        rows.append({
            "model": name,
            "seed": SEED,
            "split": "test",
            "config": json.dumps(configuration, sort_keys=True),
            **metrics,
        })
    return pd.DataFrame(rows), per_user


torch.set_num_threads(2)
comparison_bpr_config = dict(dim=64, lr=1e-3, reg=1e-4, batch_size=4096,
                             epochs=100, seed=SEED, patience=5, fused_adam=True)
bpr_variants = {
    "standard_bpr": (0, False),
    "log_weighted_bpr": (1, False),
    "user_normalized_bpr": (1, True),
}
bpr_runs = {}
comparison_jaccard = (jaccard if best_k == "all" else
                      ItemJaccard(train_df, num_users, num_items, top_k_neighbors=None))
test_models = {
    "popularity": (popularity, {}),
    "item_jaccard": (comparison_jaccard, {"top_k_neighbors": "all"}),
}
for name, (alpha, normalize) in bpr_variants.items():
    weights = playtime_weights(train_df, alpha=alpha, user_normalize=normalize)
    bpr_runs[name] = train_bpr(
        train_df, val_df, num_users, num_items, weights=weights, **comparison_bpr_config
    )
    model, history = bpr_runs[name]
    best_epoch = max(history, key=lambda row: row["val_ndcg@10"])["epoch"]
    configuration = dict(alpha=alpha, user_normalize=normalize, checkpoint_epoch=best_epoch,
                         epochs_run=len(history),
                         **comparison_bpr_config)
    test_models[name] = model, configuration

alpha_history, best_alpha, best_alpha_model = run_alpha_sweep(
    train_df, val_df, num_users, num_items,
    previous_runs={0: bpr_runs["standard_bpr"], 1: bpr_runs["log_weighted_bpr"]},
    **comparison_bpr_config,
)
if best_alpha not in (0, 1):
    selected_rows = alpha_history.loc[alpha_history["alpha"] == best_alpha]
    best_epoch = int(selected_rows.loc[selected_rows["val_ndcg@10"].idxmax(), "epoch"])
    configuration = dict(alpha=best_alpha, user_normalize=False, checkpoint_epoch=best_epoch,
                         epochs_run=len(selected_rows), **comparison_bpr_config)
    test_models["validation_selected_alpha"] = best_alpha_model, configuration

main_comparison, test_per_user = evaluate_test_models(
    test_models, train_df, val_df, test_df, num_users, num_items
)
main_comparison.to_csv(RESULTS_DIR / "main_comparison.csv", index=False)
display(main_comparison.drop(columns=["seed", "split", "config"]).round(4))

### 4.2 Experiment B: playtime sensitivity (alpha sweep, validation)  <!-- [T] -->

Sweep $\alpha\in\{0,0.25,0.5,1,2\}$ for log-weighted BPR using the same train/validation split, seed, 100-epoch cap, and patience of five. Select the best validation checkpoint for each setting by NDCG@10, breaking ties in favor of the smallest $\alpha$. The sweep runs before test evaluation in Section 4.1.

Validation NDCG@10 declined as $\alpha$ increased across the five settings. The unweighted model ($\alpha=0$) reached 0.2364. The strongest positive exponent was $\alpha=0.25$, at 0.2329. The selection was therefore standard BPR, already included in the test comparison.

In [ ]:
alpha_history.to_csv(RESULTS_DIR / "bpr_alpha_sweep.csv", index=False)
best_epochs = alpha_history.groupby("alpha", sort=True)["val_ndcg@10"].idxmax()
alpha_summary = alpha_history.loc[best_epochs, ["alpha", "epoch", "val_ndcg@10"]]
display(alpha_summary.reset_index(drop=True))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alpha_summary["alpha"], alpha_summary["val_ndcg@10"], marker="o")
ax.set(xlabel="Playtime weight exponent (alpha)", ylabel="Validation NDCG@10")
ax.set_xticks(alpha_summary["alpha"])
fig.tight_layout()
fig.savefig(FIGURES_DIR / "bpr_alpha_sweep.png", dpi=160)
plt.show()

### 4.3 Experiment C: user activity groups  <!-- [S+T] -->

We order users by total training playtime, with ties broken by increasing user ID. The bottom and top quarters are the low- and high-activity groups. We report each group's test metrics separately to check whether weighting affects users with different amounts of recorded playtime.

Standard BPR led both weighted variants on test NDCG@10 in all three activity groups. The drop from log weighting was 0.0250 in the bottom quarter, 0.0202 in the middle half, and 0.0159 in the top quarter. Total training playtime correlates with library size, so the group differences cannot isolate the effect of playtime alone.

In [ ]:
def activity_group_metrics(train_df, test_per_user, model_names):
    """Macro-average model metrics within training-playtime groups."""
    hours = train_df.groupby("user_id", sort=True)["playtime_hours"].sum()
    ordered_users = hours.sort_values(kind="stable").index
    quarter = len(ordered_users) // 4
    labels = ["Bottom 25%", "Middle 50%", "Top 25%"]
    sizes = [quarter, len(ordered_users) - 2 * quarter, quarter]
    groups = pd.Series(
        pd.Categorical(np.repeat(labels, sizes), categories=labels, ordered=True),
        index=ordered_users,
        name="activity_group",
    )
    rows = []
    for model_name in model_names:
        users = test_per_user[model_name].join(groups, on="user_id")
        summary = users.groupby("activity_group", observed=True).agg(
            n_users=("user_id", "size"),
            **{metric: (metric, "mean") for metric in ("recall@10", "recall@20", "ndcg@10", "auc")},
        ).reset_index()
        rows.append(summary.assign(model=model_name, seed=SEED, split="test"))
    return pd.concat(rows, ignore_index=True)


activity_results = activity_group_metrics(train_df, test_per_user, bpr_variants)
activity_results.to_csv(RESULTS_DIR / "bpr_user_activity_test.csv", index=False)
display(activity_results.drop(columns=["seed", "split"]).round(4))

**Playtime or library size?** Users with more total playtime also tend to own more games, so we repeat the comparison with users grouped by number of training games. If the gap between weighted and standard BPR follows library size as closely as playtime, the group differences cannot be attributed to playtime alone.

In [ ]:
# Is the pattern about playtime or about library size? Total playtime and library size are
# correlated (Section 2.4), so repeat the grouping by number of training games, using the same
# rule as activity_group_metrics (stable sort with ties by user ID, quarter = n // 4).
def quartile_groups(values, name):
    ordered_users = values.sort_values(kind="stable").index
    quarter = len(ordered_users) // 4
    labels = ["Bottom 25%", "Middle 50%", "Top 25%"]
    sizes = [quarter, len(ordered_users) - 2 * quarter, quarter]
    return pd.Series(pd.Categorical(np.repeat(labels, sizes), categories=labels, ordered=True),
                     index=ordered_users, name=name).sort_index()


by_user = train_df.groupby("user_id", sort=True)
playtime_group = quartile_groups(by_user["playtime_hours"].sum(), "playtime group")
library_group = quartile_groups(by_user["item_id"].size(), "library-size group")

# How often do the two groupings agree?
display(pd.crosstab(playtime_group, library_group))
print(f"Users in the same group under both groupings: {(playtime_group == library_group).mean():.0%}")

# NDCG@10 of each BPR variant under both groupings, and the weighted-minus-standard gap.
ndcg_by_group = {}
for grouping in (playtime_group, library_group):
    for model_name in bpr_variants:
        per_user = test_per_user[model_name].join(grouping, on="user_id")
        ndcg_by_group[(grouping.name, model_name)] = (
            per_user.groupby(grouping.name, observed=True)["ndcg@10"].mean())
ndcg_by_group = pd.DataFrame(ndcg_by_group).T
ndcg_by_group.index.names = ["grouping", "model"]
for weighted in ("log_weighted_bpr", "user_normalized_bpr"):
    for grouping in (playtime_group.name, library_group.name):
        ndcg_by_group.loc[(grouping, f"{weighted} - standard"), :] = (
            ndcg_by_group.loc[(grouping, weighted)] - ndcg_by_group.loc[(grouping, "standard_bpr")])
display(ndcg_by_group.sort_index().round(4))

Only 57% of users fall in the same group under both groupings, so the two can be told apart. The log-weighting gap narrows from the bottom to the top quarter under both: from −0.025 to −0.016 NDCG@10 by playtime, but only from −0.022 to −0.019 by library size. The gap therefore tracks playtime somewhat more closely than library size. This fits the explanation in Section 4.5: heavy players have fewer never-played games among their held-out games, so down-weighting unplayed games costs them less.

### 4.4 Effect of extending the BPR training budget (validation)  <!-- [T] -->

This section compares the best checkpoints from the first 30 epochs with the best checkpoints from the full early-stopped runs in Section 4.1. The shared evaluator verified the restored checkpoints across all 56,457 validation users.

Longer training improved validation NDCG@10 for all three variants. Standard BPR led with 0.2364 at epoch 91, up from 0.2138 at epoch 30. The test and activity results in Sections 4.1 and 4.3 use these same extended checkpoints.

In [ ]:
extended_summaries, extended_histories = [], []
for name, (model, history) in bpr_runs.items():
    alpha, _ = bpr_variants[name]
    best = max(history, key=lambda row: row["val_ndcg@10"])
    previous = max(history[:30], key=lambda row: row["val_ndcg@10"])
    metrics = evaluate(model, val_df, [train_df], num_users, num_items, batch_size=512)
    np.testing.assert_allclose(metrics["ndcg@10"], best["val_ndcg@10"], rtol=0, atol=1e-12)
    run = dict(model=name, split="validation", alpha=alpha, **comparison_bpr_config,
               train_interactions=len(train_df), validation_interactions=len(val_df),
               torch_threads=torch.get_num_threads())
    extended_summaries.append({
        **run,
        **metrics,
        "epoch": best["epoch"],
        "epochs_run": len(history),
        "early_stopped": len(history) < comparison_bpr_config["epochs"],
        "previous_best_epoch": previous["epoch"],
        "previous_ndcg@10": previous["val_ndcg@10"],
        "ndcg@10_gain": metrics["ndcg@10"] - previous["val_ndcg@10"],
    })
    extended_histories.append(pd.DataFrame(history).assign(**run))

extended_results = pd.DataFrame(extended_summaries)
extended_history = pd.concat(extended_histories, ignore_index=True)
extended_results.to_csv(RESULTS_DIR / "bpr_extended_validation.csv", index=False)
extended_history.to_csv(RESULTS_DIR / "bpr_extended_training_history.csv", index=False)

extended_labels = {
    "standard_bpr": "Standard BPR",
    "log_weighted_bpr": "Log-weighted BPR",
    "user_normalized_bpr": "User-normalized BPR",
}
extended_table = extended_results[
    ["model", "previous_ndcg@10", "ndcg@10", "epoch", "epochs_run"]
].copy()
extended_table["model"] = extended_table["model"].map(extended_labels)
display(extended_table.rename(columns={
    "model": "Model",
    "previous_ndcg@10": "30-epoch validation NDCG@10",
    "ndcg@10": "Extended validation NDCG@10",
    "epoch": "Best epoch",
    "epochs_run": "Stopped at",
}).round(4))

fig, ax = plt.subplots(figsize=(8, 4.5))
for (name, label), color in zip(extended_labels.items(), ["#22568b", "#af5927", "#3b7d68"]):
    history = extended_history[extended_history["model"] == name]
    best = history.loc[history["val_ndcg@10"].idxmax()]
    ax.plot(history["epoch"], history["val_ndcg@10"], color=color, label=label)
    ax.scatter(best["epoch"], best["val_ndcg@10"], color=color, zorder=3, s=28)
ax.axvline(30, color="#606060", ls="--", lw=1, label="Previous 30-epoch cap")
ax.set(xlabel="Training epoch", ylabel="Validation NDCG@10")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "bpr_extended_validation.png", dpi=160)
plt.show()

### 4.5 Played versus never-played held-out games  <!-- [S] -->

About a third of the held-out test games were never played, yet each counts as a hit if it is recommended. If playtime weighting hurts mainly because it pushes training away from unplayed games, the weighted models should lose most of their ground on held-out games the user never played. We measure, for each model, the share of held-out games that reach the user's top 20, separately for played and never-played games. Test playtime is used only to group the results; no model sees it.

In [ ]:
def heldout_hits_by_playtime(model, k=20, batch_size=512):
    """For each held-out test game, whether it appears in the user's top k (same masking and
    tie-breaking as evaluate). Returns a boolean array aligned with test_df rows."""
    masked = interaction_matrix([train_df, val_df], num_users, num_items)
    users = np.unique(test_df["user_id"].to_numpy())
    top_rows, top_cols = [], []
    for start in range(0, len(users), batch_size):
        batch = users[start:start + batch_size]
        scores = model.score_users(batch)
        if hasattr(scores, "detach"):
            scores = scores.detach().cpu().numpy()
        scores = np.array(scores, dtype=np.float64)
        scores[masked[batch].toarray().astype(bool)] = -np.inf
        top = np.argsort(-scores, axis=1, kind="stable")[:, :k]
        top_rows.append(np.repeat(batch, k))
        top_cols.append(top.ravel())
    in_top = sp.csr_matrix((np.ones(sum(len(r) for r in top_rows), dtype=np.int8),
                            (np.concatenate(top_rows), np.concatenate(top_cols))),
                           shape=(num_users, num_items))
    return np.asarray(in_top[test_df["user_id"].to_numpy(), test_df["item_id"].to_numpy()]).ravel() > 0


# Played vs never-played held-out games. Test playtime is only used to group the results here;
# no model sees it. Hit rate = share of held-out games of that kind that reach the top 20.
played = test_df["playtime_hours"].to_numpy() > 0
print(f"Held-out test games never played: {(~played).mean():.1%} of {len(played):,}")
playtime_split = []
for name, (model, _) in test_models.items():
    hits = heldout_hits_by_playtime(model, k=20)
    playtime_split.append({"model": name,
                           "hit rate@20 (played)": hits[played].mean(),
                           "hit rate@20 (never played)": hits[~played].mean(),
                           "hit rate@20 (all)": hits.mean()})
playtime_split = pd.DataFrame(playtime_split).set_index("model")
playtime_split.to_csv(RESULTS_DIR / "heldout_hits_by_playtime.csv")
display(playtime_split.round(4))

**Playtime weighting does what it is designed to do, but the task rewards something else.** On held-out games the user *played*, both weighted variants beat standard BPR (hit rate@20 0.297 and 0.293 against 0.281). On held-out games the user *never played*, they fall far behind (0.163 and 0.158 against 0.237). Never-played games make up 35% of the test targets, so the loss there outweighs the gain on played games, and the weighted models lose overall.

Weighting therefore improves the ranking of games a user will engage with, at the cost of games they merely own. Whether that is better depends on the goal. For predicting ownership, as evaluated here, it is worse. For recommending games people will actually play, it may be better, and evaluating against played games only would be a natural next experiment. Popularity shows the same pattern in a more extreme form: it finds played games far more easily than unplayed ones (0.189 against 0.071), while Jaccard is nearly balanced.

## 5. Related work and limitations  <!-- [S+T] -->

### How this dataset has been used

- **Pathak, Gupta and McAuley (2017)** collected this dataset by crawling the members of the Australian "GameAus" Steam community, together with Steam's bundle offers, to study bundle recommendation. Their first stage is item-level BPR with matrix factorization, evaluated by AUC, which is close to our standard BPR. They note that a small number of games account for most purchases, and that uniformly sampled negatives therefore bias the model toward popular games. Their fix was to draw negatives with the same popularity distribution as positives. [Generating and Personalizing Bundle Recommendations on Steam](https://cseweb.ucsd.edu/~jmcauley/pdfs/sigir17.pdf)
- **Wan and McAuley (2018)** use the same Steam data to model *monotonic behavior chains*: a stronger action implies the weaker ones, so on Steam a recommendation implies a review, which implies playing, which implies a purchase. Their model, chainRec, predicts each stage of the chain rather than treating all interactions alike, and is evaluated with AUC and NDCG. They report that about 64% of purchased games were played. [Item Recommendation on Monotonic Behavior Chains](https://cseweb.ucsd.edu/~jmcauley/pdfs/recsys18b.pdf)
- **Kang and McAuley (2018)** use the larger Version 2 Steam review data for sequential recommendation (predicting a user's next game from the order of their reviews). Our snapshot has no timestamps, so this setting is not available to us. [Self-Attentive Sequential Recommendation](https://arxiv.org/abs/1808.09781)

### How prior work approached the task

- Rendle et al. (2009) derive a pairwise ranking objective for implicit feedback. A training triplet pairs an observed item with an unobserved item for the same user, and the loss encourages a higher score for the observed item. Our baseline applies this objective to Steam ownership with dot-product embeddings and one sampled negative per positive per epoch. [BPR: Bayesian Personalized Ranking from Implicit Feedback](https://arxiv.org/abs/1205.2618)
- Hu, Koren and Volinsky (2008) derive binary preferences from implicit observations and use interaction intensity to weight a squared-error objective by confidence. They also describe logarithmic confidence scaling. We apply the confidence idea to the pairwise ranking loss by scaling each positive game's contribution with its logged training playtime. Per-user normalization tests whether weighting games within a user's library adds signal when each user's total weight is held fixed. [Collaborative Filtering for Implicit Feedback Datasets](https://yifanhu.net/PUB/cf.pdf)
- **Lerche and Jannach (2014)** extend BPR to graded implicit feedback, so that stronger signals are ranked above weaker ones rather than treated as equal positives. They report accuracy gains on e-commerce datasets. Our weighting is a simpler variant of the same idea, so we do not claim it as a new method. [Using Graded Implicit Feedback for Bayesian Personalized Ranking](https://web-ainf.aau.at/pub/jannach/files/Conference_RecSys2014a.pdf)

### How our results compare

- **BPR versus the baselines.** Standard BPR was the best model on every test metric (NDCG@10 0.279, AUC 0.977), ahead of item-based Jaccard (0.270, 0.961) and popularity (0.158, 0.950). This matches the common finding that a well-trained latent-factor model beats neighborhood and popularity baselines on implicit feedback. The margin over Jaccard is small, though, and Jaccard needs no training.
- **AUC rewards popularity.** Popularity alone reaches an AUC of 0.950 with full-catalog uniform candidates, because most unowned games are obscure. This is the skew Pathak et al. describe, and it is why we use NDCG@10 as the primary metric: it separates the models much more clearly (0.158 to 0.279) than AUC does (0.950 to 0.977). Their AUC values were measured against popularity-matched negatives, so they are not comparable with ours.
- **Playtime weighting did not help overall.** Unlike the gains reported for confidence weighting (Hu et al.) and graded BPR (Lerche and Jannach), every weighted variant scored below standard BPR, and validation NDCG@10 fell steadily as the weight exponent grew (0.236 at $\alpha=0$ to 0.202 at $\alpha=2$). Longer training did not change this (Section 4.4).
  - *The reason is the target.* We predict ownership, and about 35% of owned games were never played (consistent with the 64% play rate Wan and McAuley report). Section 4.5 shows the weighted models rank held-out *played* games better than standard BPR, but held-out *never-played* games much worse. Hu et al. and Lerche and Jannach evaluate against the stronger signals they weight by; we evaluate against the weakest one, ownership, which playtime weighting deliberately de-emphasizes.
  - *Playtime also reflects game type.* Multiplayer and open-ended games accumulate far more hours than short single-player games a user may like just as much, so the weights partly encode genre rather than preference strength. We have not tested this.
  - Wan and McAuley's approach suggests a better use of playtime: treat *playing* as a separate, stronger behavior to predict, rather than reweighting the ownership objective.

### Limitations

- Single seed: differences of about 0.01 NDCG@10 between models may be within seed-to-seed variation.
- All BPR variants share one training configuration. This keeps the comparison controlled, but the weighted variants might do better with a different learning rate or regularization.
- The split is random rather than chronological, and held-out ownership is a proxy for preference: games a user does not own are not necessarily games they would dislike.
- The users come from a single Australian community with public profiles, and the non-game filter is a name-based heuristic.